In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
df_train = pd.read_csv('C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\save fichier rakuten\\df_train_cleaned.csv')

df_train= df_train.drop("Unnamed: 0", axis=1)

df_train.head()

In [ ]:
Y_train = pd.read_csv('C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\save fichier rakuten\\Y_train_final.csv')

Y_train= Y_train.drop("Unnamed: 0", axis=1)

Y_train.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_train['Texte'], Y_train, test_size=0.2)

In [ ]:
from keras.preprocessing.text import Tokenizer #convertir les mots en séquences d'entiers en utilisant un index de mots.

# Convertir les valeurs en chaînes de caractères
X_train = X_train.astype(str)
X_test = X_test.astype(str)

tokenizer = Tokenizer(num_words=20000) # cinsidere les 18.000 mots les plus frequent, default 10.000

tokenizer.fit_on_texts(X_train)

X_train = tokenizer.texts_to_sequences(X_train)

X_test = tokenizer.texts_to_sequences(X_test)

vocab_size = len(tokenizer.word_index) + 1  # +1 parce que l'index commence par 0

In [ ]:
from keras.preprocessing.sequence import pad_sequences # afin que les sequences aient toutes la même longueur

maxlen = 400 

X_train = pad_sequences(X_train, padding='post', maxlen=maxlen)

X_test = pad_sequences(X_test, padding='post', maxlen=maxlen)

In [ ]:
print("X_train", len(X_train))
print("X_test", len(X_test))

# DNN simple

In [ ]:
embedding_dim = 100
nbr_Epochs = 100
batch_size = 200

In [ ]:
from tensorflow.keras.layers import LSTMCell
from tensorflow.keras import Sequential, layers
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D, GlobalMaxPooling1D, RNN, GRUCell, Dropout, concatenate, SpatialDropout1D, Conv1D
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint
from keras.callbacks import EarlyStopping
from sklearn import metrics


In [ ]:
model_DNN = Sequential()
model_DNN.add(layers.Embedding(input_dim=vocab_size, 
                           output_dim=embedding_dim, 
                           input_length=maxlen))
model_DNN.add(layers.Flatten())
model_DNN.add(layers.Dense(100, activation='relu'))
model_DNN.add(layers.Dropout(0.5)) 
model_DNN.add(layers.Dense(27, activation='softmax'))

model_DNN.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])


learning_rate =  0.001

model_DNN.compile(optimizer= Adam(learning_rate= learning_rate), #  0.001 , default LR
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model_DNN.summary()

In [ ]:
%timeit
date_result='09032024'

file_name_checkpnt1 = 'checkpoint_SimpleDNN_' + date_result +  \
                                       '_LR_'+str(learning_rate) + '_' + str(nbr_Epochs)+ 'Epochs.h5'

# Sauvegarder les meilleurs poids du modèle au cours de l'entraînement :
checkpoint1 = ModelCheckpoint(filepath = 'C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\DNN\\Simple DNN\\' + file_name_checkpnt1 , 
                             monitor ='val_accuracy',
                             verbose = 1,
                             save_best_only = True,
                             save_weights_only = False,
                             mode ='max',
                             save_freq='epoch')

# Arrêter l'entraînement si le modèle n'évolue plus :
early1 = EarlyStopping(monitor='val_loss',                    
                      min_delta = 0,
                      patience =10,
                      restore_best_weights = True,
                      verbose = 1,
                      mode = 'min')

callbacks1 = [checkpoint1, early1]


In [ ]:
'C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\DNN\\Simple DNN\\' + file_name_checkpnt1

In [ ]:
history_model_DNN = model_DNN.fit(X_train, y_train.values,
                            batch_size = 200,
                            epochs = nbr_Epochs,
                            validation_data = [X_test, y_test.values],
                            callbacks = callbacks1
                           )

In [ ]:
valid_score_model_DNN = model_DNN.evaluate(X_test, y_test.values, batch_size=batch_size)

In [ ]:
y_pred_model_DNN = model_DNN.predict(X_test)
#Appliquer la méthode argmax du tableau y_pred pour obtenir les classes les plus probable

y_pred_class_model_DNN = y_pred_model_DNN.argmax(axis = 1)

In [ ]:
%%time
#Get the accuracy score
valid_score_model_DNN = model_DNN.evaluate(X_test, y_test, batch_size=batch_size)

print("Accuracy: {:.2f}%".format(valid_score_model_DNN[1] * 100)) 
print("Loss: ",valid_score_model_DNN[0])
simpDNN_SC_model_DNN = np.round((valid_score_model_DNN[1] * 100), 2)

In [ ]:
precision = precision_score(y_test.values, y_pred_class_model_DNN, average='weighted')
recall = recall_score(y_test.values, y_pred_class_model_DNN, average='weighted')
f1 = f1_score(y_test.values, y_pred_class_model_DNN, average='weighted')

print(f'Précision : {precision}')
print(f'Rappel : {recall}')
print(f'Score F1 : {f1}')


In [ ]:
print(metrics.classification_report( y_test.values, y_pred_class_model_DNN))

# Display Model loss , accuracy per epoch

In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(121)
plt.plot(history_model_DNN.history['loss'])
plt.plot(history_model_DNN.history['val_loss'])
plt.title('Model loss by epoch')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'valid'], loc='right')

plt.subplot(122)
plt.plot(history_model_DNN.history['accuracy'])
plt.plot(history_model_DNN.history['val_accuracy'])
plt.title('Model acc by epoch')
plt.ylabel('acc')
plt.xlabel('epoch')
plt.legend(['train', 'valid'], loc='right')
plt.show()

# CONV1D

In [ ]:
date_result = '09032024'
nbr_Epochs = 50
batch_size = 150

In [ ]:
model_conv1D = Sequential()

model_conv1D.add(layers.Embedding(vocab_size, embedding_dim, input_length=maxlen))
model_conv1D.add(layers.SpatialDropout1D(0.2))
model_conv1D.add(layers.Conv1D(64, 2, activation='relu'))
model_conv1D.add(layers.GlobalMaxPooling1D())
model_conv1D.add(layers.Dense(27, activation='softmax'))

         
model_conv1D.summary()

In [ ]:
learning_rate =  0.001

model_conv1D.compile(optimizer= Adam(learning_rate= learning_rate), #  0.001 , default LR
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
%%time

file_name_checkpnt = 'checkpoint_Conv1D_' + date_result +  \
                                       '_LR_'+str(learning_rate) +'_' + str(nbr_Epochs)+ 'Epochs.h5'

#Sauvegarder les meilleurs poids du modèle au cours de l'entraînement :
checkpoint = ModelCheckpoint(filepath = 'C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\DL\\CONV1D' + file_name_checkpnt , 
                             monitor ='val_accuracy',
                             verbose = 1,
                             save_best_only = True, # (pour que le meilleur modèle ne soit pas écrasé)
                             save_weights_only = False,
                             mode ='max', #(permet de préciser si la métrique doit croître ou décroitre : ici on choisit 'min'
                                           #car la métrique est une perte à minimiser)
                             save_freq='epoch')
#Arrêter l'entraînement si le modèle n'évolue plus (très pratique pour ne pas gérer le
#nombre d'epoch) :

early = EarlyStopping(monitor='val_loss',                    
                      min_delta = 0,
                      patience = 10, #(nombre d'epochs à attendre avant d'arrêter l'entraînement
                      restore_best_weights = True,
                      verbose = 1,
                      mode = 'min')


callbacks = [checkpoint, early]

In [ ]:
history_conv1D = model_conv1D.fit(X_train, y_train.values,
    batch_size = 200,
    epochs = nbr_Epochs,
    validation_data = [X_test, y_test.values],
    callbacks = [checkpoint, early])

In [ ]:
valid_score_conv1D = model_conv1D.evaluate(X_test, y_test.values, batch_size=batch_size)

In [ ]:
y_pred_conv1D = model_conv1D.predict(X_test)

#Appliquer la méthode argmax du tableau y_pred pour obtenir les classes les plus probable

y_pred_class_conv1D = y_pred_conv1D.argmax(axis = 1)

In [ ]:
%%time
#Get the accuracy score
valid_score = model_conv1D.evaluate(X_test, y_test, batch_size=batch_size)

print("Accuracy: {:.2f}%".format(valid_score[1] * 100)) 
print("Loss: ",valid_score[0])
Conv1D_SC = np.round((valid_score[1] * 100), 2)

In [ ]:
precision = precision_score(y_test.values, y_pred_class_conv1D, average='weighted')
recall = recall_score(y_test.values, y_pred_class_conv1D, average='weighted')
f1 = f1_score(y_test.values, y_pred_class_conv1D, average='weighted')

print(f'Précision : {precision}')
print(f'Rappel : {recall}')
print(f'Score F1 : {f1}')


In [ ]:
print(metrics.classification_report( y_test.values, y_pred_class_conv1D))

In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(121)
plt.plot(history_conv1D.history['loss'])
plt.plot(history_conv1D.history['val_loss'])
plt.title('Model loss by epoch')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'valid'], loc='right')

plt.subplot(122)
plt.plot(history_conv1D.history['accuracy'])
plt.plot(history_conv1D.history['val_accuracy'])
plt.title('Model acc by epoch')
plt.ylabel('acc')
plt.xlabel('epoch')
plt.legend(['train', 'valid'], loc='right')
plt.show()